### 1. Формализация данных и постановка гипотез
Введем случайные величины $X$ (реакция на ДНХБ) и $Y$ (реакция на масло), имеющие распределение Бернулли. Пусть $n_{jk}$ — количество наблюдений исхода $(X=j, Y=k)$.
Нас интересуют несовпадающие исходы:
* $n_{01}$ — реакция на масло есть, на ДНХБ нет.
* $n_{10}$ — реакция на ДНХБ есть, на масло нет.

Сформулируем статистические гипотезы:
* **Нулевая гипотеза ($H_0$):** $p_{10} = p_{01}$ (вероятности несовпадающих исходов равны, разницы нет).
* **Альтернативная гипотеза ($H_1$):** $p_{10} \neq p_{01}$ (различия статистически значимы).

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.contingency_tables import mcnemar

# Матрица частот из условия задачи
data = np.array([[81, 48],
                 [23, 21]])

# Обернем в DataFrame для наглядности
table = pd.DataFrame(data, 
                     index=['Масло (+)', 'Масло (-)'], 
                     columns=['ДНХБ (+)', 'ДНХБ (-)'])

print("Таблица сопряженности:")
display(table)

Таблица сопряженности:


,ДНХБ (+),ДНХБ (-)
Масло (+),81,48
Масло (-),23,21


### 2. Расчет критерия Мак-Немара с поправкой Йейтса
По теореме Мак-Немара статистика сходится к распределению хи-квадрат с одной степенью свободы ($df = 1$). Формула с поправкой на непрерывность:
$$\chi^2_{Yates} = \frac{(|n_{10} - n_{01}| - 1)^2}{n_{10} + n_{01}}$$

Подставим наши значения ($n_{10} = 23$, $n_{01} = 48$):
$$\chi^2_{obs} = \frac{(|23 - 48| - 1)^2}{23 + 48} = \frac{24^2}{71} \approx 8.113$$

Сравним с критическим значением $\chi^2_{1, 0.99} \approx 6.635$ (для $\alpha = 0.01$).

In [2]:
# Извлекаем несовпадающие частоты
n01 = table.iloc[0, 1] # 48
n10 = table.iloc[1, 0] # 23

# Считаем статистику по формуле
chi2_yates = (abs(n10 - n01) - 1)**2 / (n10 + n01)

# Вычисляем p-value (df = 1) через функцию выживания (sf = 1 - cdf)
p_value_manual = stats.chi2.sf(chi2_yates, df=1)

print(f"Расчетное значение хи-квадрат: {chi2_yates:.3f}")
print(f"p-value: {p_value_manual:.5e}")

Расчетное значение хи-квадрат: 8.113
p-value: 4.39568e-03


### 3. Программная проверка через statsmodels
Реализация метода `mcnemar` с параметрами `exact=False` (аппроксимация $\chi^2$) и `correction=True` аппаратно дублирует формулу $\chi^2_{Yates}$.

In [4]:
# Передаем чистый numpy array (data), а не pandas DataFrame (table)
# Альтернативно можно написать table.to_numpy()
result = mcnemar(data, exact=False, correction=True)

print(f"Статистика хи-квадрат (statsmodels): {result.statistic:.3f}")
print(f"p-value (statsmodels): {result.pvalue:.5e}")
print("-" * 40)

# Интерпретация результата
alpha = 0.01

if result.pvalue < alpha:
    print(f"p-value < {alpha}. Отвергаем H0: Различия статистически значимы.")
else:
    print(f"p-value >= {alpha}. Не удалось отвергнуть H0: Различия случайны.")

Статистика хи-квадрат (statsmodels): 8.113
p-value (statsmodels): 4.39568e-03
----------------------------------------
p-value < 0.01. Отвергаем H0: Различия статистически значимы.
